In [2]:
import os

from sionna.mapping import Constellation, Mapper, Demapper
from sionna.fec.ldpc import LDPC5GEncoder, LDPC5GDecoder
from sionna.utils import ebnodb2no
from sionna.channel import AWGN, FlatFadingChannel

import tensorflow as tf
from PIL import Image
import math
import numpy as np

In [3]:
def imBatchtoImage(batch_images):
    '''
    turns b, 32, 32, 3 images into single sqrt(b) * 32, sqrt(b) * 32, 3 image.
    '''
    batch, h, w, c = batch_images.shape
    b = int(batch ** 0.5)

    divisor = b
    while batch % divisor != 0:
        divisor -= 1
    
    image = tf.reshape(batch_images, (-1, batch//divisor, h, w, c))
    image = tf.transpose(image, [0, 2, 1, 3, 4])
    image = tf.reshape(image, (-1, batch//divisor*w, c))
    return image

In [4]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

BATCH_SIZE = 64

def dataset_generator(dir, mode=None, shuffle=True):
    if mode:
        dataset = image_dataset_from_directory(
            directory=dir,
            label_mode='int',
            labels='inferred',
            color_mode='rgb',
            batch_size=BATCH_SIZE,
            image_size=(32, 32),
            shuffle=shuffle,
            interpolation='bilinear',
            validation_split=0.1,
            subset=mode,
            seed=0
        )
    else:
        dataset = image_dataset_from_directory(
            directory=dir,
            label_mode='int',
            labels='inferred',
            color_mode='rgb',
            batch_size=BATCH_SIZE,
            image_size=(32, 32),
            shuffle=shuffle,
            interpolation='bilinear'
        )

    return dataset

In [5]:
test_ds = dataset_generator('/home/vboxuser/SwinJSCC_implementation/dataset/raw/CIFAR-10-images/test')

Found 10000 files belonging to 10 classes.


2026-05-21 08:35:59.643443: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [16]:
!/home/vboxuser/SwinJSCC_implementation/src/libbpg-0.9.1/bpgenc

BPG Image Encoder version 0.9.1
usage: bpgenc [options] infile.[jpg|png]

Main options:
-h                   show the full help (including the advanced options)
-o outfile           set output filename (default = out.bpg)
-q qp                set quantizer parameter (smaller gives better quality,
                     range: 0-51, default = 28)
-f cfmt              set the preferred chroma format (420, 422, 444,
                     default=420)
-c color_space       set the preferred color space (ycbcr, rgb, ycgco,
                     default=ycbcr)
-b bit_depth         set the bit depth (8 to 12, default = 10)
-lossless            enable lossless mode


In [ ]:
class BPGEncoder():
    def __init__(self, working_directory='/home/vboxuser/SwinJSCC_implementation/analysis/temp'):
        '''
        working_directory: directory to save temp files
                           do not include '/' in the end
        '''
        self.working_directory = working_directory
    
    def run_bpgenc(self, qp, input_dir, output_dir='temp.bpg'):
        if os.path.exists(output_dir):
            os.remove(output_dir)
        os.system(f'/home/vboxuser/SwinJSCC_implementation/src/libbpg-0.9.1/bpgenc {input_dir} -q {qp} -o {output_dir} -f 444')

        if os.path.exists(output_dir):
            return os.path.getsize(output_dir)
        else:
            return -1
    
    def get_qp(self, input_dir, byte_threshold, output_dir='temp.bpg'):
        '''
        iteratively finds quality parameter that maximizes quality given the byte_threshold constraint
        '''
        # rate-match algorithm
        quality_max = 51
        quality_min = 0
        quality = (quality_max - quality_min) // 2
        
        while True:
            qp = 51 - quality
            bytes = self.run_bpgenc(qp, input_dir, output_dir)
            if quality == 0 or quality == quality_min or quality == quality_max:
                break
            elif bytes > byte_threshold and quality_min != quality - 1:
                quality_max = quality
                quality -= (quality - quality_min) // 2
            elif bytes > byte_threshold and quality_min == quality - 1:
                quality_max = quality
                quality -= 1
            elif bytes < byte_threshold and quality_max > quality:
                quality_min = quality
                quality += (quality_max - quality) // 2
            else:
                break
        
        return qp
    
    def encode(self, image_array, max_bytes, header_bytes=22):
        '''
        image_array: uint8 numpy array with shape (b, h, w, c)
        max_bytes: int, maximum bytes of the encoded image file (exlcuding header bytes)
        header_bytes: the size of BPG header bytes (to be excluded in image file size calculation)
        '''

        input_dir = f'{self.working_directory}/temp_enc.png'
        output_dir = f'{self.working_directory}/temp_enc.bpg'

        im = Image.fromarray(image_array, 'RGB')
        im.save(input_dir)

        qp = self.get_qp(input_dir, max_bytes + header_bytes, output_dir)
        
        if self.run_bpgenc(qp, input_dir, output_dir) < 0:
            raise RuntimeError("BPG encoding failed")

        # read binary and convert it to numpy binary array with float dtype
        return np.unpackbits(np.fromfile(output_dir, dtype=np.uint8)).astype(np.float32)


In [7]:
class LDPCTransmitter():
    '''
    Transmits given bits (float array of '0' and '1') with LDPC.
    '''
    def __init__(self, k, n, m, esno_db, channel='AWGN'):
        '''
        k: data bits per codeword (in LDPC)
        n: total codeword bits (in LDPC)
        m: modulation order (in m-QAM)
        esno_db: channel SNR
        channel: 'AWGN' or 'Rayleigh'
        '''
        self.k = k
        self.n = n
        self.num_bits_per_symbol = round(math.log2(m))

        constellation_type = 'qam' if m != 2 else 'pam'
        self.constellation = Constellation(constellation_type, num_bits_per_symbol=self.num_bits_per_symbol)
        self.mapper = Mapper(constellation=self.constellation)
        self.demapper = Demapper('app', constellation=self.constellation)
        self.channel = AWGN() if channel == 'AWGN' else FlatFadingChannel
        self.encoder = LDPC5GEncoder(k=self.k, n=self.n)
        self.decoder = LDPC5GDecoder(self.encoder, num_iter=20)
        self.esno_db = esno_db
    

    def send(self, source_bits):
        '''
        source_bits: float np array of '0' and '1', whose total # of bits is divisible with k
        '''
        lcm = np.lcm(self.k, self.num_bits_per_symbol)
        source_bits_pad = tf.pad(source_bits, [[0, math.ceil(len(source_bits)/lcm)*lcm - len(source_bits)]])
        u = np.reshape(source_bits_pad, (-1, self.k))

        no = ebnodb2no(self.esno_db, num_bits_per_symbol=1, coderate=1)
        c = self.encoder(u)
        x = self.mapper(c)
        y = self.channel([x, no])
        llr_ch = self.demapper([y, no])
        u_hat = self.decoder(llr_ch)

        return tf.reshape(u_hat, (-1))[:len(source_bits)]

In [8]:
class BPGDecoder():
    def __init__(self, working_directory='/home/vboxuser/SwinJSCC_implementation/analysis/temp'):
        '''
        working_directory: directory to save temp files
                           do not include '/' in the end
        '''
        self.working_directory = working_directory
    
    def run_bpgdec(self, input_dir, output_dir='temp.png'):
        if os.path.exists(output_dir):
            os.remove(output_dir)
        os.system(f'/home/vboxuser/SwinJSCC_implementation/src/libbpg-0.9.1/bpgdec {input_dir} -o {output_dir}')

        if os.path.exists(output_dir):
            return os.path.getsize(output_dir)
        else:
            return -1

    def decode(self, bit_array, image_shape):
        '''
        returns decoded result of given bit_array.
        if bit_array is not decodable, then returns the mean CIFAR-10 pixel values.

        byte_array: float array of '0' and '1'
        image_shape: used to generate image with mean pixel values if the given byte_array is not decodable
        '''
        input_dir = f'{self.working_directory}/temp_dec.bpg'
        output_dir = f'{self.working_directory}/temp_dec.png'

        byte_array = np.packbits(bit_array.astype(np.uint8))
        with open(input_dir, "wb") as binary_file:
            binary_file.write(byte_array.tobytes())

        cifar_mean = np.array([0.4913997551666284, 0.48215855929893703, 0.4465309133731618]) * 255
        cifar_mean = np.reshape(cifar_mean, [1] * (len(image_shape) - 1) + [3]).astype(np.uint8)

        if self.run_bpgdec(input_dir, output_dir) < 0:
            # print('warning: Decode failed. Returning mean pixel value')
            return 0 * np.ones(image_shape) + cifar_mean
        else:
            x = np.array(Image.open(output_dir).convert('RGB'))
            if x.shape != image_shape:
                return 0 * np.ones(image_shape) + cifar_mean
            return x


In [9]:
import tensorflow as tf
print(tf.__version__)

2.11.0


In [10]:
import tensorflow as tf
import numpy as np

def create_window(window_size=3, sigma=1.5, channel=3):
    coords = tf.range(window_size, dtype=tf.float32)
    coords -= window_size // 2

    g = tf.exp(-(coords ** 2) / (2 * sigma ** 2))
    g /= tf.reduce_sum(g)

    g = tf.reshape(g, [1, window_size, 1, 1])

    g = tf.repeat(g, channel, axis=2)

    return g

def gaussian_filter(x, window, use_padding=False):
    padding = 'SAME' if use_padding else 'VALID'

    out = tf.nn.depthwise_conv2d(x, window, strides=[1, 1, 1, 1], padding=padding)

    window_t = tf.transpose(window, [1, 0, 2, 3])

    out = tf.nn.depthwise_conv2d(out, window_t, strides=[1, 1, 1, 1], padding=padding)

    return out

def ssim_tf(X, Y, window, data_range=255.0, use_padding=False):

    K1 = 0.01
    K2 = 0.03

    C1 = (K1 * data_range) ** 2
    C2 = (K2 * data_range) ** 2

    mu1 = gaussian_filter(X, window, use_padding)
    mu2 = gaussian_filter(Y, window, use_padding)

    sigma1_sq = gaussian_filter(X * X, window, use_padding)
    sigma2_sq = gaussian_filter(Y * Y, window, use_padding)
    sigma12 = gaussian_filter(X * Y, window, use_padding)

    mu1_sq = mu1 ** 2
    mu2_sq = mu2 ** 2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = sigma1_sq - mu1_sq
    sigma2_sq = sigma2_sq - mu2_sq
    sigma12 = sigma12 - mu1_mu2

    cs_map = (2.0 * sigma12 + C2) / (sigma1_sq + sigma2_sq + C2)
    cs_map = tf.nn.relu(cs_map)

    ssim_map = (((2.0 * mu1_mu2 + C1) / (mu1_sq + mu2_sq + C1)) * cs_map)

    ssim_val = tf.reduce_mean(ssim_map, axis=[1, 2, 3])
    cs = tf.reduce_mean(cs_map, axis=[1, 2, 3])

    return ssim_val, cs

def ms_ssim_tf(X, Y, window_size=3, sigma=1.5, data_range=255.0, levels=4, weights=None, se_padding=False, eps=1e-8):

    if weights is None:
        weights = [0.0448, 0.2856, 0.3001, 0.2363]
    else:
        weights = weights[:levels]

    weights = tf.constant(weights, dtype=tf.float32)

    channel = X.shape[-1]

    window = create_window(
        window_size=window_size,
        sigma=sigma,
        channel=channel
    )

    vals = []

    for i in range(levels):

        ss, cs = ssim_tf(X, Y, window, data_range=data_range, use_padding=use_padding)

        if i < levels - 1:
            vals.append(cs)

            # Downsample
            X = tf.nn.avg_pool2d(X, ksize=2, strides=2, padding='SAME')

            Y = tf.nn.avg_pool2d(Y, ksize=2, strides=2, padding='SAME')

        else:
            vals.append(ss)

    vals = tf.stack(vals, axis=0)
    vals = tf.maximum(vals, eps)

    weights = tf.reshape(weights, [-1, 1])

    ms_ssim_val = tf.reduce_prod(vals ** weights, axis=0)

    return ms_ssim_val

In [ ]:
from tqdm.auto import tqdm
import math
import tensorflow as tf

bpgencoder = BPGEncoder()
bpgdecoder = BPGDecoder()

bw_ratio = [1/24]
snrs = [0]

# BPSK, QPSK, 64-QAM
mcs = [(k, n, m) for k, n in [(4096, 6144)] for m in (4, 64)]

for esno_db in snrs:
    for bw in bw_ratio:
        for k, n, m in mcs:

            psnr = 0.0
            ms_ssim_total = 0.0
            total_images = 0

            ldpctransmitter = LDPCTransmitter(k, n, m, esno_db, 'AWGN')

            for image, _ in tqdm(test_ds):
                b, _, _, _ = image.shape

                image = tf.cast(imBatchtoImage(image), tf.uint8)

                max_bytes = (b * 32 * 32 * 3 * bw * math.log2(m) * k / n / 8)

                src_bits = bpgencoder.encode(image.numpy(), max_bytes)

                rcv_bits = ldpctransmitter.send(src_bits)

                decoded_image = bpgdecoder.decode(rcv_bits.numpy(), image.shape)

                decoded_image = tf.cast(decoded_image, tf.float32)

                image = tf.cast(
                    image,
                    tf.float32
                )

                if len(decoded_image.shape) == 3:
                    decoded_image = tf.expand_dims(
                        decoded_image,
                        axis=0
                    )

                if len(image.shape) == 3:
                    image = tf.expand_dims(
                        image,
                        axis=0
                    )

                batch_psnr = tf.reduce_mean(tf.image.psnr(decoded_image, image, max_val=255.0))

                batch_ms_ssim = ms_ssim_tf(
                    decoded_image,
                    image,
                    window_size=3,
                    levels=4,
                    data_range=255.0
                )

                batch_ms_ssim = tf.reduce_mean(batch_ms_ssim)

                total_images += b

                psnr = (((total_images - b) * psnr) + float(batch_psnr) * b) / total_images

                ms_ssim_total = (((total_images - b) * ms_ssim_total) + float(batch_ms_ssim) * b) / total_images

            print(f'SNR={esno_db}, bw={bw}, k={k}, n={n}, m={m}, PSNR={psnr:.2f}, MS-SSIM={ms_ssim_total:.4f}')

  0%|          | 0/157 [00:00<?, ?it/s]

Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not d

SNR=0, bw=0.041666666666666664, k=4096, n=6144, m=4, PSNR=12.08, MS-SSIM=0.1608


  0%|          | 0/157 [00:00<?, ?it/s]

Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not decode image
Could not d

SNR=0, bw=0.041666666666666664, k=4096, n=6144, m=64, PSNR=12.08, MS-SSIM=0.1606


Could not decode image


In [ ]:
# BPG + Capacity
from tqdm.auto import tqdm

bpgencoder = BPGEncoder()
bpgdecoder = BPGDecoder()

snrs = [4, 10, 18]
mcs = [(1, 1, 1)] # dummy
bw_ratio = [1/8]
'''
(3072, 6144), (3072, 4608), (1536, 4608)
BPSK, 4-QAM, 16-QAM, 64-QAM
'''

for esno_db in snrs:
    for bw in bw_ratio:
        for _, _, _ in mcs:
            i = 0
            psnr = 0
            ssim = 0
            total_images = 0
            for image, _ in tqdm(test_ds):
                b, _, _, _ = image.shape
                image = tf.cast(imBatchtoImage(image), tf.uint8)
                max_bytes = b * 32 * 32 * 3 * bw * math.log2(1 + 10 ** (esno_db/10)) / 8
                src_bits = bpgencoder.encode(image.numpy(), max_bytes)
                decoded_image = bpgdecoder.decode(src_bits, image.shape)
                total_images += b
                psnr = (total_images - b) / (total_images) * psnr + float(b * tf.image.psnr(decoded_image, image, max_val=255)) / (total_images)
                ssim = (total_images - b) / (total_images) * ssim + float(b * tf.image.ssim(tf.cast(decoded_image, dtype=tf.float32), tf.cast(image, dtype=tf.float32), max_val=255)) / (total_images)

            print(f'SNR={esno_db},bw={bw},PSNR={psnr:.2f},SSIM={ssim:.2f}')


  0%|          | 0/10 [00:00<?, ?it/s]

SNR=0,PSNR=21.66,SSIM=0.72


  0%|          | 0/10 [00:00<?, ?it/s]

SNR=10,PSNR=26.82,SSIM=0.89


  0%|          | 0/10 [00:00<?, ?it/s]

...

SNR=10,PSNR=37.69,SSIM=0.99


  0%|          | 0/10 [00:00<?, ?it/s]

SNR=0,PSNR=30.49,SSIM=0.95


  0%|          | 0/10 [00:00<?, ?it/s]

SNR=10,PSNR=42.29,SSIM=1.00


In [1]:
# BPG + LDPC (1/6 BW ratio)
from tqdm.auto import tqdm

bpgencoder = BPGEncoder()
bpgdecoder = BPGDecoder()

bw_ratio = [1/36]
snrs = [15]
mcs = [(k, n, m) for k, n in [(4096, 6144)] for m in (2, 4, 64)]
'''
(3072, 6144), (3072, 4608), (1536, 4608)
BPSK, 4-QAM, 16-QAM, 64-QAM
'''

for esno_db in snrs:
    for bw in bw_ratio:
        for k, n, m in mcs:
            i = 0
            psnr = 0
            ssim = 0
            total_images = 0
            ldpctransmitter = LDPCTransmitter(k, n, m, esno_db, 'AWGN')
            for image, _ in tqdm(test_ds):
                b, _, _, _ = image.shape
                image = tf.cast(imBatchtoImage(image), tf.uint8)
                max_bytes = b * 32 * 32 * 3 * bw * math.log2(m) * k / n / 8
                src_bits = bpgencoder.encode(image.numpy(), max_bytes)
                rcv_bits = ldpctransmitter.send(src_bits)
                
                decoded_image = bpgdecoder.decode(rcv_bits.numpy(), image.shape)
                total_images += b
                psnr = (total_images - b) / (total_images) * psnr + float(b * tf.image.psnr(decoded_image, image, max_val=255)) / (total_images)
                ssim = (total_images - b) / (total_images) * ssim + float(b * tf.image.ssim(tf.cast(decoded_image, dtype=tf.float32), tf.cast(image, dtype=tf.float32), max_val=255)) / (total_images)

            print(f'SNR={esno_db},bw={bw},k={k},n={n},m={m},PSNR={psnr:.2f},SSIM={ssim:.2f}')


NameError: name 'BPGEncoder' is not defined

In [ ]:
# BPG + Capacity
from tqdm.auto import tqdm

bpgencoder = BPGEncoder()
bpgdecoder = BPGDecoder()

bw_ratio = [1/24]
snrs = [0, 2, 5, 7, 10, 12, 15]
mcs = [(1,1,1)] # dummy
'''
(3072, 6144), (3072, 4608), (1536, 4608)
BPSK, 4-QAM, 16-QAM, 64-QAM
'''

for esno_db in snrs:
    for bw in bw_ratio:
        for _, _, _ in mcs:
            i = 0
            psnr = 0
            ssim = 0
            total_images = 0
            for image, _ in tqdm(test_ds):
                b, _, _, _ = image.shape
                image = tf.cast(imBatchtoImage(image), tf.uint8)
                max_bytes = b * 32 * 32 * 3 * bw * math.log2(1 + 10 ** (esno_db/10)) / 8
                src_bits = bpgencoder.encode(image.numpy(), max_bytes)
                decoded_image = bpgdecoder.decode(src_bits, image.shape)
                total_images += b
                psnr = (total_images - b) / (total_images) * psnr + float(b * tf.image.psnr(decoded_image, image, max_val=255)) / (total_images)
                ssim = (total_images - b) / (total_images) * ssim + float(b * tf.image.ssim(tf.cast(decoded_image, dtype=tf.float32), tf.cast(image, dtype=tf.float32), max_val=255)) / (total_images)

            print(f'SNR={esno_db},bw={bw},PSNR={psnr:.2f},SSIM={ssim:.2f}')


  0%|          | 0/157 [00:00<?, ?it/s]

SNR=0,bw=0.125,PSNR=23.21,SSIM=0.78


  0%|          | 0/157 [00:00<?, ?it/s]

SNR=2,bw=0.125,PSNR=24.66,SSIM=0.83


  0%|          | 0/157 [00:00<?, ?it/s]

...

SNR=7,bw=0.125,PSNR=28.70,SSIM=0.92


  0%|          | 0/157 [00:00<?, ?it/s]

SNR=10,bw=0.125,PSNR=31.49,SSIM=0.96


  0%|          | 0/157 [00:00<?, ?it/s]

RuntimeError: BPG encoding failed